In [43]:
!pip -q install -U datasets evaluate seqeval transformers accelerate sentencepiece huggingface_hub
!pip -q install -U natasha slovnet razdel
!pip -q install -U gliner
!pip -q install -U scikit-learn seaborn matplotlib
!pip -q install -U optimum onnxruntime

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gliner 0.2.26 requires transformers<5.2.0,>=4.51.3, but you have transformers 5.8.0 which is incompatible.


In [3]:
import os, re, math, random, gc

import numpy as np
import pandas as pd

import torch
from torch import nn
import torch.nn.functional as F

from datasets import load_dataset, DatasetDict
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    pipeline
)

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix

from natasha import Doc, Segmenter, NewsEmbedding, NewsNERTagger
from razdel import sentenize

from gliner import GLiNER

from huggingface_hub import login
from google.colab import userdata

In [31]:
def bio_to_spans(labels):
    spans = []
    start = None
    ent_type = None

    for i, lab in enumerate(labels):
        if lab == "O":
            if ent_type is not None:
                spans.append((start, i, ent_type))
                start, ent_type = None, None
            continue

        prefix, typ = lab.split("-", 1)
        if prefix == "B":
            if ent_type is not None:
                spans.append((start, i, ent_type))
            start, ent_type = i, typ
        elif prefix == "I":
            if ent_type is None:
                # некорректная последовательность: считаем как B
                start, ent_type = i, typ
            elif typ != ent_type:
                spans.append((start, i, ent_type))
                start, ent_type = i, typ

    if ent_type is not None:
        spans.append((start, len(labels), ent_type))
    return spans

In [4]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [42]:
ds = load_dataset("gusevski/factrueval2016")
ds["train"][0]
ds

Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['data'],
        num_rows: 1
    })
    validation: Dataset({
        features: ['data'],
        num_rows: 1
    })
    test: Dataset({
        features: ['data'],
        num_rows: 1
    })
})

In [44]:
ds["train"][0]

{'data': [{'id': 0,
   'tokens': ['"',
    'Если',
    'Миронов',
    'занял',
    'столь',
    'оппозиционную',
    'позицию',
    ',',
    'то',
    'мне',
    'представляется',
    ',',
    'что',
    'для',
    'него',
    'было',
    'бы',
    'порядочным',
    'и',
    'правильным',
    'уйти',
    'в',
    'отставку',
    'с',
    'занимаемого',
    'им',
    'поста',
    ',',
    'поста',
    ',',
    'который',
    'предоставлен',
    'ему',
    'сегодня',
    '"',
    'Единой',
    'Россией',
    "''",
    'и',
    'никем',
    'больше',
    "''",
    ',',
    '-',
    'заключает',
    'Исаев',
    '.'],
   'length': 47,
   'ner_tags_str': ['O',
    'O',
    'B-PER',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'B-ORG',
    'I-ORG',


In [45]:
from datasets import Dataset, DatasetDict

ds = DatasetDict({
    "train": Dataset.from_list(ds["train"][0]["data"]),
    "validation": Dataset.from_list(ds["validation"][0]["data"]),
    "test": Dataset.from_list(ds["test"][0]["data"]),
})

ds

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 7746
    })
    validation: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 2582
    })
    test: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 2582
    })
})

In [46]:
ds["train"][0]

{'id': 0,
 'tokens': ['"',
  'Если',
  'Миронов',
  'занял',
  'столь',
  'оппозиционную',
  'позицию',
  ',',
  'то',
  'мне',
  'представляется',
  ',',
  'что',
  'для',
  'него',
  'было',
  'бы',
  'порядочным',
  'и',
  'правильным',
  'уйти',
  'в',
  'отставку',
  'с',
  'занимаемого',
  'им',
  'поста',
  ',',
  'поста',
  ',',
  'который',
  'предоставлен',
  'ему',
  'сегодня',
  '"',
  'Единой',
  'Россией',
  "''",
  'и',
  'никем',
  'больше',
  "''",
  ',',
  '-',
  'заключает',
  'Исаев',
  '.'],
 'length': 47,
 'ner_tags_str': ['O',
  'O',
  'B-PER',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-ORG',
  'I-ORG',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-PER',
  'O'],
 'ner_tags': [0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0

In [47]:
ds["train"].features

{'id': Value('int64'),
 'tokens': List(Value('string')),
 'length': Value('int64'),
 'ner_tags_str': List(Value('string')),
 'ner_tags': List(Value('int64'))}

In [48]:
ds["train"].features["ner_tags"]

List(Value('int64'))

In [49]:
label_names = sorted(
    list(set(
        tag
        for example in ds["train"]
        for tag in example["ner_tags_str"]
    ))
)

label_names

['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']

In [50]:
train_size = 5000
val_size = 1000
test_size = 1000

ds_small = DatasetDict({
    "train": ds["train"].shuffle(seed=SEED).select(range(train_size)),
    "validation": ds["validation"].shuffle(seed=SEED).select(range(val_size)),
    "test": ds["test"].shuffle(seed=SEED).select(range(test_size)),
})
ds_small

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 5000
    })
    validation: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 1000
    })
})

### Утилиты для оценки качества

In [14]:
seqeval_metric = evaluate.load("seqeval")

def seqeval_strict_micro_f1(y_true, y_pred):
    # evaluate/seqeval дает entity-level строгое совпадение спана
    # В res есть overall_f1/precision/recall
    res = seqeval_metric.compute(predictions=y_pred, references=y_true, zero_division=0)
    return res

In [15]:
def boundary_error_breakdown(y_true, y_pred):
    # Анализ по спанам: exact, type_confusion, boundary_mismatch, missing, spurious
    stats = {
        "exact": 0,
        "type_confusion": 0,
        "boundary_mismatch": 0,
        "missing": 0,
        "spurious": 0,
        "total_gold": 0,
        "total_pred": 0,
    }

    for gt, pr in zip(y_true, y_pred):
        gsp = bio_to_spans(gt)
        psp = bio_to_spans(pr)

        stats["total_gold"] += len(gsp)
        stats["total_pred"] += len(psp)

        gset = {(s, e, t) for (s, e,  t) in gsp}
        pset = {(s, e, t) for (s, e, t) in psp}

        # Точные совпадения - одинаковые start, end и type
        exact = gset & pset
        stats["exact"] += len(exact)

        # Те же границы, разный тип
        gb = {(s, e): t for (s, e, t) in gsp}
        pb = {(s, e): t for (s, e, t) in psp}
        for b in set(gb.keys()) & set(pb.keys()):
            if gb[b] != pb[b]:
                stats["type_confusion"] += 1

        # boundary mismatch - есть пересечение по токенам, но точного совпадения по границам нет
        def overlaps(a, b):
            (s1, e1, _t1) = a
            (s2, e2, _t2) = b
            return max(s1, s2) < min(e1, e2)

        # Для каждой gold-сущности, которая не попала в exact:
        # если она пересекается хотя бы с одним предсказанным спаном, это boundary_mismatch
        # если не пересекается ни с одним, это missing
        for g in gsp:
            if g in exact:
                continue
            if any(overlaps(g, p) for p in psp):
                stats["boundary_mismatch"] += 1
            else:
                stats["missing"] += 1

        # Для каждой предсказанной сущности, которая не попала в exact:
        # если она не пересекается ни с одной gold-сущностью, это spurious
        # если пересекается, ничего не добавляется, потому что boundary_mismatch
        # уже считают только по gold-стороне, чтобы не удваивать счётчик
        for p in psp:
            if p in exact:
                continue
            if any(overlaps(p, g) for g in gsp):
                pass
            else:
                stats["spurious"] += 1

    return stats

In [16]:
def confusion_matrix_by_type(y_true, y_pred, entity_types=("PER","ORG","LOC")):
    # Матрица только по exact boundary matches + type confusion (по одинаковым границам)
    # Классы: entity_types
    labels = list(entity_types)
    idx = {t:i for i,t in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=int)

    for gt, pr in zip(y_true, y_pred):
        gsp = bio_to_spans(gt)
        psp = bio_to_spans(pr)
        gb = {(s,e): t for (s,e,t) in gsp}
        pb = {(s,e): t for (s,e,t) in psp}
        for b in set(gb.keys()) & set(pb.keys()):
            g = gb[b]
            p = pb[b]
            if g in idx and p in idx:
                cm[idx[g], idx[p]] += 1

    return cm, labels

def plot_confusion(cm, labels, title):
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.xlabel("pred")
    plt.ylabel("gold")
    plt.show()

#### deepvk/RuModernBERT-small as-is

In [51]:
small_model_id = "deepvk/RuModernBERT-small"

tokenizer = AutoTokenizer.from_pretrained(small_model_id)

label_names = sorted(
    list(set(
        tag
        for example in ds_small["train"]
        for tag in example["ner_tags_str"]
    ))
)

id2label = {i: l for i, l in enumerate(label_names)}
label2id = {l: i for i, l in enumerate(label_names)}

загрузили токенизатор, создали маппинг лэйблов в цифры для модели

In [52]:
def tokenize_and_align_labels(examples, max_length=256):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=max_length
    )

    aligned_labels = []
    for i in range(len(examples["tokens"])):
        word_ids = tokenized.word_ids(batch_index=i)
        gold = examples["ner_tags"][i]
        prev = None
        out = []
        for w in word_ids:
            if w is None:
                out.append(-100)
            elif w != prev:
                out.append(gold[w])
            else:
                # стандартно: продолжения слова игнорируем
                out.append(-100)
            prev = w
        aligned_labels.append(out)

    tokenized["labels"] = aligned_labels
    return tokenized

max_length = 192
tokenized = ds_small.map(lambda x: tokenize_and_align_labels(x, max_length=max_length), batched=True)
tokenized

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 5000
    })
    validation: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
})

Разбили слова на subwords, выровняли лэйблы. Появились input_ids (токены модели в виде чисе), attention_mask - настоящие и пустые токены для выранивания размерности модели, выровненные labels после tokenizer. Теперь такой датасет поймет transformer

In [22]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    y_true, y_pred = [], []
    for p, l in zip(preds, labels):
        true_labels = []
        pred_labels = []
        for pi, li in zip(p, l):
            if li == -100:
                continue
            true_labels.append(id2label[li])
            pred_labels.append(id2label[pi])
        y_true.append(true_labels)
        y_pred.append(pred_labels)

    res = seqeval_metric.compute(predictions=y_pred, references=y_true, zero_division=0)
    return {
        "f1": res["overall_f1"],
        "precision": res["overall_precision"],
        "recall": res["overall_recall"],
    }

Считает качество через seqeval

In [53]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding="longest")

model = AutoModelForTokenClassification.from_pretrained(
       small_model_id,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

# Freeze encoder на быстрый разогрев головы
for p in model.base_model.parameters():
    p.requires_grad = False

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

ModernBertForTokenClassification LOAD REPORT from: deepvk/RuModernBERT-small
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


То есть: attention не обучается, embeddings не меняются, transformer weights фиксированы. обучается теперь NER classification head.

In [54]:
small_model = AutoModelForTokenClassification.from_pretrained(
    small_model_id,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

ModernBertForTokenClassification LOAD REPORT from: deepvk/RuModernBERT-small
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


То есть encoder уже предобучен, а слой, который решает: токен → PER / ORG / LOC / O пока пустой и его надо обучить. AutoModelForTokenClassification - модель для классификации токенов, А NER - token classification task (каждому токену дать label).

In [55]:
args = TrainingArguments(
    output_dir="rumodernbert_ner_ft_small",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    fp16=False,
    eval_strategy="steps",
    save_strategy="steps",
    logging_steps=50,
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0
)

trainer_small = Trainer(
    model=small_model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)



warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [56]:
preds_before = trainer_small.predict(tokenized["test"])

metrics_before = preds_before.metrics

metrics_before

{'test_loss': 2.861685037612915,
 'test_model_preparation_time': 0.003,
 'test_f1': 0.38965755793842966,
 'test_precision': 0.4467283542630535,
 'test_recall': 0.34551681832123504,
 'test_runtime': 4.9635,
 'test_samples_per_second': 201.471,
 'test_steps_per_second': 6.447}

F1 = 0.389 оценка до обучения. Предсказывает 0,4 меток

In [ ]:
trainer_warmup.train()

In [27]:
preds = trainer_small.predict(tokenized["test"])
test_metrics = preds.metrics
test_metrics

{'test_loss': 0.0722353383898735,
 'test_f1': 0.9851738241308793,
 'test_precision': 0.9852745679517333,
 'test_recall': 0.9850731009099274,
 'test_runtime': 2.3402,
 'test_samples_per_second': 427.318,
 'test_steps_per_second': 13.674}

'test_f1': 0.985 - предсказывает 98,5 меток

In [32]:
def trainer_predictions_to_seqeval(pred_output):
    logits = pred_output.predictions
    labels = pred_output.label_ids
    preds = np.argmax(logits, axis=-1)

    y_true, y_pred = [], []
    for p, l in zip(preds, labels):
        true_labels = []
        pred_labels = []
        for pi, li in zip(p, l):
            if li == -100:
                continue
            true_labels.append(id2label[li])
            pred_labels.append(id2label[pi])
        y_true.append(true_labels)
        y_pred.append(pred_labels)
    return y_true, y_pred

y_true_ft_small, y_pred_ft_small = trainer_predictions_to_seqeval(preds)

seqeval_strict_micro_f1(y_true_ft_small, y_pred_ft_small)
boundary_error_breakdown(y_true_ft_small, y_pred_ft_small)

{'exact': 19270,
 'type_confusion': 67,
 'boundary_mismatch': 283,
 'missing': 9,
 'spurious': 5,
 'total_gold': 19562,
 'total_pred': 19558}

очень мало missing, очень мало spurious, значит иногда неточно определяет границы многословных сущностей, но сами сущности находит хорошо

In [37]:
def summarize(model_name, y_true, y_pred):
    res = seqeval_metric.compute(
        predictions=y_pred,
        references=y_true,
        zero_division=0
    )

    return {
        "model": model_name,
        "f1": res["overall_f1"],
        "precision": res["overall_precision"],
        "recall": res["overall_recall"]
    }

In [38]:
rows = []
rows.append(summarize("RuModernBERT-small", y_true_ft_small, y_pred_ft_small))

In [39]:
pd.DataFrame(rows)

,model,f1,precision,recall
0,RuModernBERT-small,0.985174,0.985275,0.985073


In [57]:
from transformers import AutoModelForMaskedLM, DataCollatorForWholeWordMask

In [76]:
mlm_ds = ds_small.map(
    lambda x: {
        "text": [" ".join(tokens) for tokens in x["tokens"]]
    },
    batched=True
)

mlm_ds

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags', 'text'],
        num_rows: 5000
    })
    validation: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags', 'text'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags', 'text'],
        num_rows: 1000
    })
})

In [85]:
mlm_tokenized = mlm_ds.map(
    lambda x: tokenizer(
        x["text"],
        truncation=True,
        padding="max_length",
        max_length=192,
        return_offsets_mapping=True,
        return_special_tokens_mask=True
    ),
    batched=True,
    remove_columns=mlm_ds["train"].column_names,
    load_from_cache_file=False
)

mlm_tokenized["train"][0].keys()

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask', 'special_tokens_mask', 'offset_mapping'])

In [86]:
wwm_collator = DataCollatorForWholeWordMask(
    tokenizer=tokenizer,
    mlm_probability=0.15
)

/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:1028: FutureWarning: DataCollatorForWholeWordMask is deprecated and will be removed in a future version, you can now use DataCollatorForLanguageModeling with whole_word_mask=True instead.
  warnings.warn(


In [87]:
mlm_model = AutoModelForMaskedLM.from_pretrained(small_model_id)

Loading weights:   0%|          | 0/77 [00:00<?, ?it/s]

In [88]:
mlm_args = TrainingArguments(
    output_dir="rumodernbert_small_mlm_wwm",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    fp16=False,
    eval_strategy="steps",
    save_strategy="steps",
    logging_steps=50,
    eval_steps=200,
    save_steps=200,
    save_total_limit=1,
    report_to="none",
    weight_decay=0.01,
    remove_unused_columns=False
)

In [89]:
mlm_trainer = Trainer(
    model=mlm_model,
    args=mlm_args,
    train_dataset=mlm_tokenized["train"],
    eval_dataset=mlm_tokenized["validation"],
    processing_class=tokenizer,
    data_collator=wwm_collator
)

mlm_trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Step,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=157, training_loss=5.821594165388945, metrics={'train_runtime': 109.3363, 'train_samples_per_second': 45.73, 'train_steps_per_second': 1.436, 'total_flos': 87830323200000.0, 'train_loss': 5.821594165388945, 'epoch': 1.0})

сделали MLM дообучение с Whole Word Masking

In [92]:
mlm_trainer.save_model("rumodernbert_small_mlm_wwm")
tokenizer.save_pretrained("rumodernbert_small_mlm_wwm")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('rumodernbert_small_mlm_wwm/tokenizer_config.json',
 'rumodernbert_small_mlm_wwm/tokenizer.json')

In [93]:
small_model_mlm_wwm = AutoModelForTokenClassification.from_pretrained(
    "rumodernbert_small_mlm_wwm",
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

for p in small_model_mlm_wwm.base_model.parameters():
    p.requires_grad = False

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

ModernBertForTokenClassification LOAD REPORT from: rumodernbert_small_mlm_wwm
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [94]:
trainer_mlm_wwm_ner = Trainer(
    model=small_model_mlm_wwm,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer_mlm_wwm_ner.train()

Step,Training Loss,Validation Loss,F1,Precision,Recall
200,1.203421,0.530750,0.852093,0.845036,0.859269
400,0.652596,0.320965,0.898889,0.893294,0.904555
600,0.526988,0.269862,0.913267,0.908315,0.918273
800,0.456316,0.245485,0.919788,0.915517,0.924100
1000,0.451644,0.232168,0.922275,0.916963,0.927648
1200,0.423484,0.225009,0.924994,0.920218,0.929820
1400,0.398875,0.221843,0.926157,0.921327,0.931038


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1570, training_loss=0.7582743492855388, metrics={'train_runtime': 113.8804, 'train_samples_per_second': 439.057, 'train_steps_per_second': 13.786, 'total_flos': 425865217717632.0, 'train_loss': 0.7582743492855388, 'epoch': 10.0})

In [95]:
preds_mlm_wwm = trainer_mlm_wwm_ner.predict(tokenized["test"])
metrics_mlm_wwm = preds_mlm_wwm.metrics
metrics_mlm_wwm

{'test_loss': 0.21457722783088684,
 'test_f1': 0.92904340754295,
 'test_precision': 0.923784494086728,
 'test_recall': 0.934362539617626,
 'test_runtime': 3.2595,
 'test_samples_per_second': 306.8,
 'test_steps_per_second': 9.818}

In [97]:
y_true_mlm_wwm, y_pred_mlm_wwm = trainer_predictions_to_seqeval(
    preds_mlm_wwm
)

seqeval_strict_micro_f1(
    y_true_mlm_wwm,
    y_pred_mlm_wwm
)

boundary_error_breakdown(
    y_true_mlm_wwm,
    y_pred_mlm_wwm
)

{'exact': 18278,
 'type_confusion': 612,
 'boundary_mismatch': 1266,
 'missing': 18,
 'spurious': 59,
 'total_gold': 19562,
 'total_pred': 19786}

In [98]:
rows = []

rows.append(
    summarize(
        "RuModernBERT-small",
        y_true_ft_small,
        y_pred_ft_small
    )
)

rows.append(
    summarize(
        "RuModernBERT-small + MLM + WWM",
        y_true_mlm_wwm,
        y_pred_mlm_wwm
    )
)

pd.DataFrame(rows)

,model,f1,precision,recall
0,RuModernBERT-small,0.985174,0.985275,0.985073
1,RuModernBERT-small + MLM + WWM,0.929043,0.923784,0.934363


MLM с Whole Word Masking не дал улчшения. Как я поняла эксперимент был в том, что бы закрывать маскированием слова, предполагая, что модель станет умнее и лучше предугадывать то, что скрывается за маской. Но это не произощло. Я бы в этой задаче скорее смотрела в сторону возможности создавтаь свои лэйблы или может вычленять такие лэйблы, что бы например сортировать почту по контрагентам или др характеристикам